# Rung 06 — LoRA on the ViT (the B probe)

**One variable against rung 02: the LoRA adapters reach the vision tower.**

The question: the model reports ~1.7 objects when there are 2, identically on ID (1.704) and
OOD (1.698), and identically in two formats that share nothing — a bare integer and a list of
class names. That is not counting and it is not output format: it is multiplicity, upstream of
both. rung 02's LoRA never touched the visual path. Does letting it reach the ViT unlock it?

**The rule is pre-registered in `local/specs/vit-lora/spec.md` and read by a human.**
This notebook reports numbers. It does not decide anything.

🔴 **Do not judge this run with `acc_number`** — it averages eight templates whose trivial floors
run from 0.24 to 1.00, four of them degenerate (`experiments/08-data-card/` §3). The target is
`dice@2` per (format × distribution), compared by a **paired** delta over videos.

## Order

`SMOKE = True` first — 2 steps, minutes. G1 and G5 fire in both modes: a flag that no-ops in
silence is how rung 07 lost a day, and G1 is the only thing separating "the LoRA reached the ViT"
from "the flag evaporated".

In [ ]:
SMOKE = True

In [ ]:
import sys
from pathlib import Path

EXP_DIR = Path.cwd()
REPO = EXP_DIR
while REPO != REPO.parent and not ((REPO / ".git").exists() or (REPO / "src").is_dir()):
    REPO = REPO.parent
for p in (EXP_DIR / "_models", REPO / "src"):
    if p.is_dir():
        sys.path.insert(0, str(p))

from vit_lora_train import ViTLoRAConfig, main, diff_vs_rung02, list_checkpoints, merge_checkpoint

cfg = ViTLoRAConfig(
    exp_dir=REPO / "experiments" / "06-vit-lora",
    smoke=SMOKE,
)
cfg.val_jsonl = cfg.run_dir / "val.jsonl"   # G-loss: observability only, no early stop
print(f"run_dir  {cfg.run_dir}")
print(f"SMOKE    {SMOKE}")

In [ ]:
# G5 — one variable. Not a promise: rung 02's argv is captured from its own engine
# (subprocess.run monkeypatched) and diffed. A hand-typed copy would be a claim.
diff = diff_vs_rung02(cfg)
for flag, (r02, r06) in sorted(diff.items()):
    print(f"  {flag:16s} rung02={r02!s:8s} -> rung06={r06}")

assert set(diff) == {"--freeze_vit", "--val_dataset"}, (
    f"G5 FAILED: rung 06 differs from rung 02 in {sorted(diff)}. Only --freeze_vit (the"
    " variable) and --val_dataset (declared observability) may differ — anything else means"
    " this run answers a different question than the pre-registered one."
)
print("\nOK G5: one variable + the declared observability")

In [ ]:
# Export. rung 02's OWN exporter — the data leg of the A/B cannot drift.
# Frames come from the shared /workspace/frames_cache (1.7 GB, already populated):
# a frame exists ONCE and is called from there. rung 05 duplicated 2.6 GB by ignoring this.
main(cfg, stage="export")
print(f"train.jsonl -> {cfg.train_jsonl} ({sum(1 for _ in open(cfg.train_jsonl))} examples)")

In [ ]:
# G-loss — the val set for eval_loss. THE_MAP:169 has asked for this since Jul 14:
# "add a val_dataset to swift sft (track eval_loss — we currently have NO val curve)".
# The `NO ERA EL TECHO` branch of the rule cites a loss that did not exist until now.
#
# Held out from OUR val split, never carved from train: verified on ms-swift 4.4.1 that
# split_dataset_ratio defaults to 0.0 and is bypassed when --val_dataset is passed, so the
# training data is byte-identical to rung 02's.
import json

from frame.config import BaselineConfig
from frame.data import load_frame_items, FrameProvider, frame_cache_name
from frame.engine import SYSTEM_PROMPT
from frame import split as sp

bcfg = BaselineConfig(data_root=cfg.data_root, model_path=cfg.model_path,
                      datasets=cfg.datasets, base_fps=cfg.base_fps,
                      max_pixels=cfg.max_pixels, seed=cfg.seed)
items = load_frame_items(bcfg, splits=("train", "test"))
val_items = sp.apply_split(items, sp.load_manifest(cfg.manifest_path), "val_ood")
val_items = sorted(val_items, key=lambda it: (it.dataset, it.video_id, it.frame_index))[:256]

provider = FrameProvider(bcfg)
with open(cfg.val_jsonl, "w", encoding="utf-8") as fh:
    for it in val_items:
        provider.ensure_reader(it)
        img = cfg.frames_dir / frame_cache_name(it)
        if not img.exists():
            provider.get_frame(it).save(img, quality=95)
        fh.write(json.dumps({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"<image>{it.request.question}"},
                {"role": "assistant", "content": str(it.reference.answer)},
            ],
            "images": [str(img)],
        }, ensure_ascii=False) + "\n")
provider.close()
print(f"val.jsonl -> {cfg.val_jsonl} ({len(val_items)} examples, held out, NOT carved from train)")

In [ ]:
# Train. G1's real proof is swift's own "trainable parameters" line in the log:
#   few M  = LoRA        ✅
#   100s M = fine-tune   ❌ -> ABORT, that is a different experiment (two variables)
# and the target-module list must contain vision_tower entries, which rung 02's did not.
#
# STOP RULES (spec): OOM -> STOP. Do NOT lower max_pixels/batch/LR — each is a second
# variable. Suspect G1 before the GPU. Loss diverges -> do NOT touch the LR; report it.
main(cfg, stage="train")
print(f"\ncheckpoints: {[c.name for c in list_checkpoints(cfg)]}")

In [ ]:
# 🔴 G1 — THE gate: did the LoRA actually reach the ViT?
#
# Read from swift's OWN log, which prints both numbers BEFORE the trainer starts:
#   tuner.py:168  lora_config: ...            <- the target modules
#   sft.py:174    model_parameter_info: ...   <- N M Params (M M Trainable [x%])
# So the 2-step SMOKE answers it in minutes, instead of after paying for the full run.
#
# It matters because tuner.py:93 has an early return (`if isinstance(target_modules, str)`)
# that ignores freeze_vit in silence. It does not fire on 4.4.1 — the CLI parses to a list,
# verified — but this is the runtime proof, not the assumption. A flag that no-ops quietly is
# exactly how rung 07 lost a day.
from vit_lora_train import read_g1

g1 = read_g1(cfg)
print(f"  model_parameter_info : {g1['model_parameter_info']}")
print(f"  trainable (M)        : {g1['trainable_params_M']}")
print(f"  targets vision_tower : {g1['targets_vision_tower']}")
print(f"  target_modules       : {str(g1['target_modules'])[:160]}")

assert g1["trainable_params_M"] is not None, (
    "G1 FAILED: swift never logged model_parameter_info — the run did not get as far as"
    " building the tuner. Read train.log; do not proceed."
)
assert g1["targets_vision_tower"], (
    "G1 FAILED: the LoRA targets contain NO vision_tower module. --freeze_vit false did"
    " nothing, so this run measures the rung-02 model under a new name. STOP and report:"
    " this is a finding about the flag, not a result about the ViT."
)
assert g1["trainable_params_M"] < 500, (
    f"G1 FAILED: {g1['trainable_params_M']}M trainable — that is a FINE-TUNE, not LoRA."
    " It would change two variables at once and answer nothing. ABORT."
)
print("\nOK G1: LoRA (not fine-tune), and it reaches the vision tower")
print("\n🚦 SMOKE stops here. Report G1 + G5, get the go-ahead, then set SMOKE=False.")